# Taller B3-T1 - Preprocesado de datos financieros para ML

Aplicación de bars, diferenciación fraccional, limpieza de covarianzas, triple barrera y validación cruzada temporal sobre datos intradía de criptomonedas.

**Integrantes:** Hugo Vinuesa y Rodolfo Villena  
**Fecha:** 15-03-2026


## Objetivo y plan

El objetivo de este notebook es aplicar cinco técnicas de preprocesado financiero orientadas a Machine Learning sobre datos intradía. El trabajo sigue un pipeline secuencial: construcción de barras por actividad, diferenciación fraccional, análisis y limpieza de covarianza, etiquetado por triple barrera y diseño de validación temporal con gap. En cada bloque se comparan parámetros alternativos y se acompañan los resultados con gráficas e interpretación. El foco no es entrenar un modelo final, sino justificar de forma visual y técnica cómo cambia la estructura del dato en cada etapa.


In [ ]:
# -----------------------------------------------------------------------------
# Importación de librerias y detección opcional de statsmodels.
# -----------------------------------------------------------------------------

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from statsmodels.graphics.tsaplots import plot_acf
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore")


In [ ]:
# -----------------------------------------------------------------------------
# Configuración visual global y definición de rutas/columnas.
# -----------------------------------------------------------------------------

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

BASE_PATH = Path(".")
DATA_PATH = BASE_PATH / "data"

KLINE_COLUMNS = [

    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "n_trades",
    "taker_buy_base", "taker_buy_quote", "ignore"
]

AGG_COLUMNS = [
    "agg_trade_id", "price", "qty", "first_trade_id", "last_trade_id",
    "timestamp", "is_buyer_maker", "is_best_match"
]


## Dataset utilizado

Se trabaja con datos de criptomonedas descargados de BINANCE por su alta actividad intradía y porque permiten observar cambios de microestructura con facilidad.

Para la parte multiactivo usamos velas de 1 minuto de:

+ BNBUSDT
+ BTCSTUSDT
+ ETHUSDT
+ SOLUSDT 
+ XRPUSDT. 

Para construir barras por actividad usamos datos de `aggTrades` de BTCSTUSDT, ya que contienen cada transacción agregada con precio, cantidad y timestamp. 

BTCSTUSDT se usa como activo principal para los bloques que requieren una serie unica.

Se ha cogido BTCSTUSDT en lugar de BTCUSDT porque el tamaño de los descargables era más manejable.


In [ ]:
# -----------------------------------------------------------------------------
# Carga de CSV intrad?a y trades agregados, con parseo de tipos.
# -----------------------------------------------------------------------------

assets_1m = {
    "BNBUSDT": DATA_PATH / "BNBUSDT-1m-2026-02.csv",
    "BTCUSDT": DATA_PATH / "BTCUSDT-1m-2026-02.csv",
    "ETHUSDT": DATA_PATH / "ETHUSDT-1m-2026-02.csv",
    "SOLUSDT": DATA_PATH / "SOLUSDT-1m-2026-02.csv",
    "XRPUSDT": DATA_PATH / "XRPUSDT-1m-2026-02.csv",
}


def infer_time_unit(values: pd.Series) -> str:
    numeric = pd.to_numeric(values, errors="coerce").dropna()
    if numeric.empty:
        raise ValueError("No hay timestamps validos para inferir unidad")

    sample = abs(float(numeric.iloc[len(numeric) // 2]))
    if sample >= 1e17:
        return "ns"
    if sample >= 1e14:
        return "us"
    if sample >= 1e11:
        return "ms"
    return "s"


data_1m = {}
open_time_units = {}
for symbol, file_path in assets_1m.items():
    df = pd.read_csv(file_path, header=None, names=KLINE_COLUMNS)

    if pd.isna(pd.to_numeric(df.loc[0, "open_time"], errors="coerce")):
        df = df.iloc[1:].reset_index(drop=True)

    time_unit = infer_time_unit(df["open_time"])
    open_time_units[symbol] = time_unit

    df["open_time"] = pd.to_datetime(pd.to_numeric(df["open_time"], errors="coerce"), unit=time_unit, utc=True)
    df["close_time"] = pd.to_datetime(pd.to_numeric(df["close_time"], errors="coerce"), unit=time_unit, utc=True)

    for col in ["open", "high", "low", "close", "volume", "quote_asset_volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    data_1m[symbol] = df

agg_files = sorted(DATA_PATH.glob("BTCUSDT-aggTrades-2026-03-*.csv"))
btc_trades = pd.concat([pd.read_csv(fp, header=None, names=AGG_COLUMNS) for fp in agg_files], ignore_index=True)

if pd.isna(pd.to_numeric(btc_trades.loc[0, "timestamp"], errors="coerce")):
    btc_trades = btc_trades.iloc[1:].reset_index(drop=True)

trade_time_unit = infer_time_unit(btc_trades["timestamp"])
btc_trades["timestamp"] = pd.to_datetime(pd.to_numeric(btc_trades["timestamp"], errors="coerce"), unit=trade_time_unit, utc=True)
btc_trades["price"] = pd.to_numeric(btc_trades["price"], errors="coerce")
btc_trades["qty"] = pd.to_numeric(btc_trades["qty"], errors="coerce")
btc_trades["dollar_value"] = btc_trades["price"] * btc_trades["qty"]

print("Activos 1m cargados:", list(data_1m.keys()))
print("Unidades detectadas open_time por activo:", open_time_units)
print("Unidad detectada timestamp aggTrades:", trade_time_unit)
print("Numero total de trades agregados BTCUSDT:", len(btc_trades))


In [ ]:
# -----------------------------------------------------------------------------
# Inspección inicial de calidad de datos para 1m y aggTrades.
# -----------------------------------------------------------------------------

btc_1m_raw = data_1m["BTCUSDT"].copy()

print("=== BTCUSDT 1m: head ===")
display(btc_1m_raw.head())

print("\n=== BTCUSDT 1m: shape, rango temporal, NaN y duplicados ===")
print("Shape:", btc_1m_raw.shape)
print("Rango:", btc_1m_raw["open_time"].min(), "->", btc_1m_raw["open_time"].max())
print("NaN totales:", int(btc_1m_raw.isna().sum().sum()))
print("Duplicados en open_time:", int(btc_1m_raw.duplicated(subset=["open_time"]).sum()))

freq_min = btc_1m_raw["open_time"].sort_values().diff().dropna().dt.total_seconds().div(60)
print("Frecuencia media (min):", round(freq_min.mean(), 4))

print("\n=== Trades BTCUSDT: head ===")
display(btc_trades.head())

print("\n=== Trades BTCUSDT: shape, rango temporal, NaN y duplicados ===")
print("Shape:", btc_trades.shape)
print("Rango:", btc_trades["timestamp"].min(), "->", btc_trades["timestamp"].max())
print("NaN totales:", int(btc_trades.isna().sum().sum()))
print("Duplicados en agg_trade_id:", int(btc_trades.duplicated(subset=["agg_trade_id"]).sum()))


In [ ]:
# -----------------------------------------------------------------------------
# Limpieza básica y creación de retornos logarítmicos.
# -----------------------------------------------------------------------------

for symbol, df in data_1m.items():
    # Orden temporal, deduplicado y tipado limpio antes de cualquier transformación.
    df.sort_values("open_time", inplace=True)
    df.drop_duplicates(subset=["open_time"], keep="last", inplace=True)
    df.dropna(subset=["open", "high", "low", "close", "volume"], inplace=True)
    # Retorno logarítmico para comparabilidad estadística entre activos.
    df["log_ret"] = np.log(df["close"]).diff()
    data_1m[symbol] = df.reset_index(drop=True)

btc_trades = btc_trades.sort_values("timestamp").drop_duplicates(subset=["agg_trade_id"], keep="last")
btc_trades = btc_trades.dropna(subset=["price", "qty", "timestamp"]).reset_index(drop=True)
btc_trades["dollar_value"] = btc_trades["price"] * btc_trades["qty"]

print("Limpieza completada.")
print("BTCUSDT 1m filas limpias:", len(data_1m["BTCUSDT"]))
print("BTCUSDT aggTrades filas limpias:", len(btc_trades))


Antes de aplicar técnicas avanzadas, se valida la calidad del dato: orden temporal, duplicados, valores nulos y consistencia numérica. Esta base limpia es necesaria porque cualquier sesgo en esta etapa se propaga al resto del pipeline.


In [ ]:
# -----------------------------------------------------------------------------
# Definición de funciones para construir tick, volume y dollar bars.
# -----------------------------------------------------------------------------

def _build_ohlcv_from_groups(df_trades, group_id_col):
    grouped = df_trades.groupby(group_id_col, sort=True)
    bars = grouped.agg(
        timestamp=("timestamp", "last"),
        open=("price", "first"),
        high=("price", "max"),
        low=("price", "min"),
        close=("price", "last"),
        volume=("qty", "sum"),
        dollar_value=("dollar_value", "sum"),
        n_trades=("agg_trade_id", "count"),
    ).reset_index(drop=True)
    bars["log_ret"] = np.log(bars["close"]).diff()
    return bars

def tick_bars(df_trades, n_ticks=100):
    df = df_trades.copy()
    df["bar_id"] = np.arange(len(df)) // int(n_ticks)
    return _build_ohlcv_from_groups(df, "bar_id")

def volume_bars(df_trades, vol_threshold):
    df = df_trades.copy()
    cum_vol = df["qty"].cumsum()
    df["bar_id"] = (cum_vol / float(vol_threshold)).astype(int)
    return _build_ohlcv_from_groups(df, "bar_id")

def dollar_bars(df_trades, dollar_threshold):
    df = df_trades.copy()
    cum_dollar = df["dollar_value"].cumsum()
    df["bar_id"] = (cum_dollar / float(dollar_threshold)).astype(int)
    return _build_ohlcv_from_groups(df, "bar_id")


In [ ]:
# -----------------------------------------------------------------------------
# Generación de barras y resumen estadístico comparativo.
# -----------------------------------------------------------------------------

TARGET_BARS = 250
n_ticks = max(20, len(btc_trades) // TARGET_BARS)
vol_threshold = max(1e-8, btc_trades["qty"].sum() / TARGET_BARS)
dollar_threshold = max(1e-8, btc_trades["dollar_value"].sum() / TARGET_BARS)

btc_tick = tick_bars(btc_trades, n_ticks=n_ticks)
btc_vol = volume_bars(btc_trades, vol_threshold=vol_threshold)
btc_dollar = dollar_bars(btc_trades, dollar_threshold=dollar_threshold)

summary_bars = pd.DataFrame(
    {


        "n_obs": [len(btc_tick), len(btc_vol), len(btc_dollar)],
        "ret_std": [
            np.log(btc_tick["close"]).diff().std(),
            np.log(btc_vol["close"]).diff().std(),
            np.log(btc_dollar["close"]).diff().std(),
        ],
    },
    index=["tick", "volume", "dollar"],
)

display(summary_bars)


In [ ]:
# -----------------------------------------------------------------------------
# Visualización comparativa de las tres familias de barras.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for name, df in {"Tick": btc_tick, "Volume": btc_vol, "Dollar": btc_dollar}.items():
    x = df["timestamp"]
    y = df["close"] / df["close"].iloc[0]
    axes[0, 0].plot(x, y, label=name)
axes[0, 0].set_title("Close normalizado por tipo de bar")
axes[0, 0].legend()

axes[0, 1].hist(btc_tick["log_ret"].dropna(), bins=30, alpha=0.5, label="Tick")
axes[0, 1].hist(btc_vol["log_ret"].dropna(), bins=30, alpha=0.5, label="Volume")

axes[0, 1].hist(btc_dollar["log_ret"].dropna(), bins=30, alpha=0.5, label="Dollar")
axes[0, 1].set_title("Distribución de retornos por tipo de bar")
axes[0, 1].legend()

axes[1, 0].bar(summary_bars.index, summary_bars["n_obs"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[1, 0].set_title("Número de barras generadas")

axes[1, 1].bar(summary_bars.index, summary_bars["ret_std"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[1, 1].set_title("Volatilidad estimada por tipo de bar")

plt.tight_layout()
plt.show()


## Interpretación bloque 1 (Bars)

Los tres esquemas generan particiones distintas del flujo de mercado. Las tick bars fijan el número de operaciones por barra, las volume bars fijan cantidad negociada y las dollar bars fijan valor monetario negociado. En este conjunto, las barras por actividad reducen irregularidades temporales respecto a velas fijas y permiten una representación más homogénea del mercado cuando cambia la intensidad operativa. Las diferencias en dispersión de retornos y número total de observaciones muestran que cada método induce una estructura estadística distinta.

**Elección para continuar el pipeline:** `dollar bars`, por ofrecer un equilibrio práctico entre estabilidad y sensibilidad a actividad real de mercado.


In [ ]:
# -----------------------------------------------------------------------------
# Funciones de diferenciación fraccional con ventana fija.
# -----------------------------------------------------------------------------

def get_fracdiff_weights(d, size):
    w = [1.0]
    for k in range(1, size):
        # Pesos binomiales generalizados para diferenciación de orden no entero.
        w_k = -w[-1] * (d - k + 1) / k
        w.append(w_k)
    return np.array(w[::-1])

def frac_diff_fixed_window(series, d, window=30):
    s = series.astype(float).copy()
    weights = get_fracdiff_weights(d, window)
    out = pd.Series(index=s.index, dtype=float)
    for i in range(window - 1, len(s)):
        # Convolución local con ventana fija para aproximar la diferenciación fraccional.
        out.iloc[i] = np.dot(weights, s.iloc[i - window + 1 : i + 1])
    return out


In [ ]:
# -----------------------------------------------------------------------------
# Aplicación de varios valores de d y tabla de métricas.
# -----------------------------------------------------------------------------

price_series = np.log(btc_dollar["close"]).rename("log_close")

d_values = [0.2, 0.4, 0.6, 0.8]
fd_series = {}
records_fd = []

for d in d_values:
    s_fd = frac_diff_fixed_window(price_series, d=d, window=30)
    fd_series[d] = s_fd


    aligned = pd.concat([price_series, s_fd], axis=1).dropna()
    corr_with_original = aligned.iloc[:, 0].corr(aligned.iloc[:, 1]) if len(aligned) > 0 else np.nan
    adf_pvalue = np.nan
    if HAS_STATSMODELS and aligned.shape[0] > 40:
        adf_pvalue = adfuller(aligned.iloc[:, 1].values, maxlag=1, regression="c", autolag="AIC")[1]
    records_fd.append({"d": d, "corr_original": corr_with_original, "adf_pvalue": adf_pvalue})

fd_summary = pd.DataFrame(records_fd)
display(fd_summary)


In [ ]:
# -----------------------------------------------------------------------------
# Gráficas de serie, correlación, estacionariedad y autocorrelación.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

orig_std = (price_series - price_series.mean()) / price_series.std()
axes[0, 0].plot(btc_dollar["timestamp"], orig_std, label="Original (log close)", color="black", linewidth=2)
for d in d_values:
    s = fd_series[d]
    s_std = (s - s.mean()) / s.std()
    axes[0, 0].plot(btc_dollar["timestamp"], s_std, label=f"FD d={d}", alpha=0.8)
axes[0, 0].set_title("Serie original vs diferenciacion fraccional")
axes[0, 0].legend(fontsize=9)

axes[0, 1].plot(fd_summary["d"], fd_summary["corr_original"], marker="o")
axes[0, 1].set_title("Correlacion con serie original")
axes[0, 1].set_xlabel("d")

axes[1, 0].plot(fd_summary["d"], fd_summary["adf_pvalue"], marker="o", color="tomato")
axes[1, 0].axhline(0.05, linestyle="--", color="gray", label="Umbral 5%")


axes[1, 0].set_title("p-value test ADF")
axes[1, 0].set_xlabel("d")
axes[1, 0].legend()

chosen_d = 0.4
chosen_series = fd_series[chosen_d].dropna()
if HAS_STATSMODELS and len(chosen_series) > 30:
    plot_acf(chosen_series, lags=20, ax=axes[1, 1])
    axes[1, 1].set_title(f"ACF para d={chosen_d}")
else:
    lags = range(1, 21)
    acf_vals = [chosen_series.autocorr(lag=lag) for lag in lags]
    axes[1, 1].bar(list(lags), acf_vals)
    axes[1, 1].set_title(f"Autocorrelacion manual para d={chosen_d}")

plt.tight_layout()
plt.show()


## Interpretación bloque 2 (Diferenciación fraccional)

Al aumentar `d`, la serie tiende a parecerse mas a una diferenciación clásica: se reduce memoria y suele aumentar estacionariedad. Con valores bajos de `d`, se conserva más información de largo plazo, aunque puede quedar mayor dependencia temporal. En este caso, `d=0.4` se toma como compromiso operativo para continuar el pipeline, ya que mantiene correlación útil con la serie original sin renunciar por completo a propiedades de estacionariedad.


In [ ]:
# -----------------------------------------------------------------------------
# Cálculo de covarianza/correlación y limpieza por autovalores.
# -----------------------------------------------------------------------------

returns_1m = pd.DataFrame({
    symbol: df.set_index("open_time")["log_ret"]
    for symbol, df in data_1m.items()
}).dropna(how="any")

cov_raw = returns_1m.cov()
corr_raw = returns_1m.corr()
std_vec = returns_1m.std().values

def denoise_corr_by_eigen_clipping(corr_matrix, q_ratio):
    eigvals, eigvecs = np.linalg.eigh(corr_matrix)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]


    lambda_plus = (1.0 + 1.0 / np.sqrt(q_ratio)) ** 2
    noise_mask = eigvals < lambda_plus
    eigvals_clean = eigvals.copy()
    if noise_mask.any():
        eigvals_clean[noise_mask] = eigvals[noise_mask].mean()
    corr_clean = eigvecs @ np.diag(eigvals_clean) @ eigvecs.T
    diag_inv = np.diag(1.0 / np.sqrt(np.diag(corr_clean)))
    corr_clean = diag_inv @ corr_clean @ diag_inv
    return corr_clean, eigvals, eigvals_clean

q_ratio = returns_1m.shape[0] / returns_1m.shape[1]
corr_clean_np, eigvals_raw, eigvals_clean = denoise_corr_by_eigen_clipping(corr_raw.values, q_ratio=q_ratio)
corr_clean = pd.DataFrame(corr_clean_np, index=corr_raw.index, columns=corr_raw.columns)
cov_clean_np = np.diag(std_vec) @ corr_clean_np @ np.diag(std_vec)
cov_clean = pd.DataFrame(cov_clean_np, index=cov_raw.index, columns=cov_raw.columns)


In [ ]:
# -----------------------------------------------------------------------------
# Comparativa visual de matrices y espectro antes/después.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.heatmap(corr_raw, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[0, 0])
axes[0, 0].set_title("Correlacion original")

sns.heatmap(corr_clean, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[0, 1])
axes[0, 1].set_title("Correlacion limpia (eigenvalue clipping)")

axes[1, 0].plot(np.sort(eigvals_raw)[::-1], marker="o", label="Original")
axes[1, 0].plot(np.sort(eigvals_clean)[::-1], marker="o", label="Limpia")


axes[1, 0].set_title("Espectro de autovalores")
axes[1, 0].legend()

sns.heatmap((corr_raw - corr_clean).abs(), annot=True, fmt=".3f", cmap="magma", ax=axes[1, 1])
axes[1, 1].set_title("Diferencia absoluta |corr_original - corr_limpia|")

plt.tight_layout()
plt.show()

display(cov_raw)
display(cov_clean)


## Interpretación bloque 3 (Limpieza de covarianza)

La matriz empírica de covarianza/correlación en alta frecuencia puede incorporar ruido estadístico. El clipping de autovalores reduce la influencia de componentes débiles asociadas a ruido y estabiliza la estructura de dependencia. Visualmente se observa un suavizado de la matriz y una compresión de autovalores pequeños, lo que mejora robustez para etapas posteriores de modelado o gestión de riesgo.


In [ ]:
# -----------------------------------------------------------------------------
# Función de etiquetado triple barrera y distribución de etiquetas.
# -----------------------------------------------------------------------------

def triple_barrier_labels(price, pt_sl=0.005, horizon=20):
    p = price.reset_index(drop=True).astype(float)
    n = len(p)
    labels = np.full(n, np.nan)
    t_end = np.full(n, np.nan)
    realized_ret = np.full(n, np.nan)

    for i in range(n - horizon):
        p0 = p.iloc[i]
        upper = p0 * (1 + pt_sl)
        lower = p0 * (1 - pt_sl)
        path = p.iloc[i + 1 : i + horizon + 1]
        hit_up = np.where(path.values >= upper)[0]
        hit_dn = np.where(path.values <= lower)[0]
        first_up = hit_up[0] if len(hit_up) > 0 else np.inf
        first_dn = hit_dn[0] if len(hit_dn) > 0 else np.inf

        if first_up == np.inf and first_dn == np.inf:
            end_idx = i + horizon
            labels[i] = 0
        elif first_up < first_dn:
            end_idx = i + 1 + int(first_up)
            labels[i] = 1
        else:
            end_idx = i + 1 + int(first_dn)


            labels[i] = -1

        t_end[i] = end_idx
        realized_ret[i] = (p.iloc[int(end_idx)] / p0) - 1.0

    return pd.DataFrame({"price": p, "label": labels, "event_end_idx": t_end, "realized_ret": realized_ret})

# Etiquetado sobre la serie elegida en el pipeline (dollar bars).
tb_price = btc_dollar["close"].copy().reset_index(drop=True)
tb_time = btc_dollar["timestamp"].copy().reset_index(drop=True)
thresholds = [0.003, 0.006, 0.01]
tb_results = {thr: triple_barrier_labels(tb_price, pt_sl=thr, horizon=20) for thr in thresholds}

dist_rows = []
for thr, df_tb in tb_results.items():
    counts = df_tb["label"].value_counts(dropna=True)
    total = counts.sum() if counts.sum() > 0 else 1
    dist_rows.append({
        "threshold": thr,
        "pct_-1": 100 * counts.get(-1.0, 0) / total,
        "pct_0": 100 * counts.get(0.0, 0) / total,
        "pct_+1": 100 * counts.get(1.0, 0) / total,
    })

tb_dist = pd.DataFrame(dist_rows)
display(tb_dist)


In [ ]:
# -----------------------------------------------------------------------------
# Ejemplos visuales de barreras y composición de clases.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

thr_plot = 0.006
horizon = 20
sample_idx = [20, 60, 100]

axes[0].plot(tb_time, tb_price, color="black", alpha=0.7, linewidth=1.2, label="Precio")
for i in sample_idx:
    if i + horizon >= len(tb_price):
        continue
    p0 = tb_price.iloc[i]
    up = p0 * (1 + thr_plot)
    dn = p0 * (1 - thr_plot)
    t0 = tb_time.iloc[i]


    t1 = tb_time.iloc[i + horizon]
    axes[0].hlines([up, dn], xmin=t0, xmax=t1, colors=["green", "red"], linestyles="--", linewidth=1)
    axes[0].axvline(t0, color="gray", linestyle=":", linewidth=0.8)

axes[0].set_title("Ejemplos de triple barrera (threshold=0.6%)")

tb_dist_plot = tb_dist.set_index("threshold")
tb_dist_plot[["pct_-1", "pct_0", "pct_+1"]].plot(kind="bar", stacked=True, ax=axes[1], colormap="Set2")
axes[1].set_title("Porcentaje de etiquetas por threshold")
axes[1].set_ylabel("Porcentaje")
axes[1].legend(title="Etiqueta")

plt.tight_layout()
plt.show()


## Interpretación bloque 4 (Triple barrera)

Con barreras más estrechas aumentan los eventos etiquetados por toque de precio, lo que incrementa sensibilidad pero también ruido potencial. Con barreras más amplias aparecen menos eventos de +1/-1 y crece el peso de expiraciones temporales. Este bloque confirma que el threshold controla el compromiso entre cantidad de ejemplos y selectividad de la señal.


In [ ]:
# -----------------------------------------------------------------------------
# Particiones temporales train/test con gap y tabla resumen.
# -----------------------------------------------------------------------------

def temporal_split_with_gap(df, train_ratio=0.7, gap=20):
    n = len(df)
    train_end = int(n * train_ratio)
    test_start = train_end + gap
    train_idx = np.arange(0, max(0, train_end))
    test_idx = np.arange(min(test_start, n), n)
    return train_idx, test_idx

validation_df = btc_dollar[["timestamp", "close", "log_ret"]].dropna().reset_index(drop=True)

gap_bars = 20
ratios = [0.6, 0.7, 0.8]
split_rows = []
for r in ratios:
    tr_idx, te_idx = temporal_split_with_gap(validation_df, train_ratio=r, gap=gap_bars)
    split_rows.append({"split": f"{int(r*100)}/{int((1-r)*100)}", "train_obs": len(tr_idx), "gap_obs": gap_bars, "test_obs": len(te_idx)})

splits_summary = pd.DataFrame(split_rows)
display(splits_summary)


In [ ]:
# -----------------------------------------------------------------------------
# Visualización de tamaños y esquema temporal de los splits.
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

x = np.arange(len(splits_summary))
axes[0].bar(x, splits_summary["train_obs"], label="Train", color="#4C72B0")
axes[0].bar(x, splits_summary["gap_obs"], bottom=splits_summary["train_obs"], label="Gap", color="#C44E52")
axes[0].bar(x, splits_summary["test_obs"], bottom=splits_summary["train_obs"] + splits_summary["gap_obs"], label="Test", color="#55A868")
axes[0].set_xticks(x)
axes[0].set_xticklabels(splits_summary["split"])
axes[0].set_title("Tamano de train/gap/test por split")
axes[0].legend()

n = len(validation_df)
for i, r in enumerate(ratios):


    tr_idx, te_idx = temporal_split_with_gap(validation_df, train_ratio=r, gap=gap_bars)
    tr_end = tr_idx[-1] if len(tr_idx) > 0 else -1
    te_start = te_idx[0] if len(te_idx) > 0 else n
    axes[1].hlines(i, 0, tr_end, color="#4C72B0", linewidth=8)
    axes[1].hlines(i, tr_end + 1, te_start - 1, color="#C44E52", linewidth=8)
    axes[1].hlines(i, te_start, n - 1, color="#55A868", linewidth=8)

axes[1].set_yticks(range(len(ratios)))
axes[1].set_yticklabels([f"{int(r*100)}/{int((1-r)*100)}" for r in ratios])
axes[1].set_title("Esquema temporal de particiones con gap")
axes[1].set_xlabel("Indice temporal")

plt.tight_layout()
plt.show()


## Interpretación bloque 5 (Validación cruzada temporal)

Los splits con mayor porcentaje de entrenamiento (por ejemplo 80/20) ofrecen mas datos para ajustar parámetros, pero reducen muestra de test. Splits mas equilibrados (60/40) aumentan capacidad de evaluación fuera de muestra, aunque con menor información para entrenar. El `gap` evita continuidad artificial entre train y test y reduce fuga temporal de información.


## Conclusiones

El pipeline secuencial permite mostrar cómo cada transformación modifica la estructura del dato financiero:

+ (1) las barras por actividad cambian la regularidad de muestreo
+ (2) la diferenciación fraccional ajusta el equilibrio memoria-estacionariedad
+ (3) la limpieza de covarianza estabiliza dependencias multiactivo
+ (4) la triple barrera define etiquetas sensibles al riesgo/beneficio
+ (5) la validación temporal con gap mejora realismo de evaluación